# Synthetic biomarker benchmark — VascX

[VascX](../docs/projects/vascx.md) is the one implementation in this catalogue that shares no code
with any other. PVBM is its own lineage, the three AutoMorph projects descend from retipy, and
OCULAR imports PVBM outright — VascX reimplements every biomarker, as configurable objects with
documented parameters rather than fixed formulas.

It is also the only one that **works in physical units**, because it is the only one that takes a
scale. Every length it returns is in millimetres.

The shapes and the values their geometry requires are in
[the shapes notebook](biomarker-synthetic-shapes.ipynb).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def repository() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError(f"nothing above {Path.cwd()} looks like the fundus-atlas repository")


ROOT = repository()
sys.path.insert(0, str(ROOT / "src"))
RESULTS = ROOT / "results" / "biomarker-synthetic" / "vascx"

from biomarkers.utils import catalogue  # noqa: E402

DECLARED = catalogue.load("vascx").declare()
NAMES = DECLARED["names"]

rows = pd.concat(
    [pd.read_csv(path) for path in sorted(RESULTS.glob("*.csv"))], ignore_index=True
)

found = []
for _, row in rows.iterrows():
    for name in rows.columns:
        if not name.startswith("said_"):
            continue
        key = name[len("said_") :]
        said, theory = row[name], row.get(f"theory_{key}")
        if pd.isna(said) or pd.isna(theory):
            continue
        found.append(
            {
                "shape": row["shape"],
                "rotation": row["rotation"],
                "metrics": key,
                "said": float(said),
                "theory": float(theory),
                "relative error": (float(said) - float(theory)) / float(theory),
            }
        )
compared = pd.DataFrame(found)

print(f"{len(rows)} renderings · {len(compared)} comparable measurements")
print(f"{len(DECLARED['keys'])} columns, {sum(1 for k in DECLARED['keys'] if '/' in k)} catalogued")
print(f"feature set: {DECLARED['feature_set']} · circle: {DECLARED['circle']}")

32 renderings · 81 comparable measurements
20 columns, 8 catalogued
feature set: fs_od_centered · circle: 1.1667 disc diameters from the disc centre


## 1. What it gets right, and by how little it misses

Two families land within seven-tenths of a per cent of what the geometry requires, on every shape
and at every angle.

In [2]:
summary = (
    compared.groupby("metrics")["relative error"]
    .agg(samples="count", mean="mean", worst=lambda e: e.abs().max())
    .sort_values("worst")
)
summary.style.format({"mean": "{:.2%}", "worst": "{:.2%}"})

,samples,mean,worst
metrics,,,
central-retinal-equivalents/knudtson/artery,8,0.67%,0.68%
central-retinal-equivalents/knudtson/vein,8,0.69%,0.69%
vessel-calibre/mean-width/vein,8,0.64%,0.69%
vessel-calibre/mean-width/artery,8,0.69%,0.69%
tortuosity/hart-tau1/artery,25,-4.32%,17.25%
tortuosity/hart-tau1/vein,24,-2.87%,17.37%


**The calibre and the central retinal equivalents are the most accurate numbers this benchmark has
measured from anybody.** Both read about 0.7% high, on every shape, in one direction — the signature
of a mask a fraction of a pixel wider than the vessel drawn, and nothing else.

For scale, on the same shapes: PVBM's Knudtson equivalent is 6.3% low on arteries, AutoMorphalyzer's
0.5% low, and AutoMorphClass and AutoMorph do not compute one at all. On calibre, AutoMorphalyzer
and AutoMorphClass read 23% and 25% high at worst; VascX reads 0.7%.

*Our finding, 2026-09-22, from reading `cre.py`:* VascX's `CRE` class combines pairs as
`c·√(d₁² + d₂²)` with c = 0.88 for arteries and 0.95 for veins — **which is Knudtson's formula**,
not Hubbard's, despite `docs/projects/vascx.md` §6 describing it as a "Hubbard reduction". Hubbard's
carries fitted constants and an additive term, and neither appears in the code. It is mapped to the
Knudtson name here on the strength of the formula rather than the label.

## 2. Tortuosity: exact on straight vessels, short on curved ones

Every other implementation in this catalogue **over**-reports tortuosity, because it measures a
digitised path by counting pixel steps and weighting a diagonal as √2. VascX does the opposite.

In [3]:
tortuosity = compared[compared["metrics"].str.startswith("tortuosity")]
tortuosity.pivot_table(
    index="shape", columns="metrics", values="relative error", aggfunc="mean"
).style.format("{:.2%}", na_rep="—")

metrics,tortuosity/hart-tau1/artery,tortuosity/hart-tau1/vein
shape,,
arc,-9.92%,—
artery-vein-pair,-0.00%,-0.00%
disjoint,-0.01%,0.01%
sinusoid,-17.18%,-17.31%
spokes-disc-centred,0.03%,0.05%
spokes-macula-centred,0.03%,0.01%
straight,0.08%,0.02%


**On anything straight it is exact to a rounding error** — 0.08% on a straight vessel, 0.03% on the
spokes, and 0.00% on the pair and the disjoint lines. No other implementation here is within four
per cent of that: PVBM reads 7.3% high on a straight vessel at 30°, AutoMorphalyzer 4.5%,
AutoMorphClass 3.7%.

**On curved vessels it reads short** — 9.9% on the arc, 17.2% on the sinusoid. That is the cost of
how it gets the straight case right: it fits splines to the vessel and caps each segment at 0.2 disc
diameters, so it smooths, and a smoothed curve is shorter than the curve it was fitted to. The
shipped set carries three caps (0.15, 0.2 and 0.25 disc diameters) and the evidence stores all
three, so a reader can see how little the cap changes the answer.

**The two errors are not interchangeable.** A chain code over-reports every vessel and worst those
at 26.6° to the grid, so it depends on the camera's orientation; a spline under-reports in
proportion to how curved the vessel actually is, which is the quantity being measured. Neither is
noise, and they point opposite ways — so a study switching between these implementations would see
tortuosity move for two unrelated reasons at once.

## 3. What is measured and not compared, and why

In [4]:
uncatalogued = [key for key in DECLARED["keys"] if "/" not in key]
pd.DataFrame(
    {"column": uncatalogued, "VascX calls it": [NAMES[k] for k in uncatalogued]}
).set_index("column")

,VascX calls it
column,
lw_tort_dist_max_segment_len_0p15_crcl_multiplier_1p16666666667_full_arteries,lw_tort_dist_max_segment_len_0p15_crcl_multipl...
lw_tort_dist_max_segment_len_0p25_crcl_multiplier_1p16666666667_full_arteries,lw_tort_dist_max_segment_len_0p25_crcl_multipl...
vd_crcl_multiplier_1p16666666667_full_arteries,vd_crcl_multiplier_1p16666666667_full_arteries
median_temporal_angle_arteries,median_temporal_angle_arteries
lw_tort_dist_max_segment_len_0p15_crcl_multiplier_1p16666666667_full_veins,lw_tort_dist_max_segment_len_0p15_crcl_multipl...
lw_tort_dist_max_segment_len_0p25_crcl_multiplier_1p16666666667_full_veins,lw_tort_dist_max_segment_len_0p25_crcl_multipl...
vd_crcl_multiplier_1p16666666667_full_veins,vd_crcl_multiplier_1p16666666667_full_veins
median_temporal_angle_veins,median_temporal_angle_veins
disc_fovea_distance_retina,disc_fovea_distance_retina


Three reasons, and they are different.

- **It is measured against an axis this fixture invented.** VascX's grids are oriented on the
  disc-to-fovea axis, and a synthetic shape has no macula. The adapter supplies a fovea from the
  framing — the frame centre, which is what *macula-centred* means and the point every shape is
  rotated about — so anything measured *superior*, *inferior*, *temporal* or *nasal* to that axis,
  and the axis itself, is a property of this repository's choice rather than of VascX. Those are
  measured, stored, and compared against nothing.
- **The catalogue has no name for it.** Its vessel density is measured over a disc-centred circle,
  which is neither the field of view nor the whole frame, so neither catalogued density describes
  it. Its sparsity is a mean where the catalogued variant is a maximum.
- **The shapes pin no value.** Two of its three tortuosity caps are kept for comparison with the
  third rather than against a theory.

## 4. What this analysis cannot say

- **Nothing about photographs.** Every shape here is clean, binary and noiseless, and VascX's
  advantage on calibre may be larger or smaller on a segmentation with a ragged boundary.
- **Nothing about most of what VascX computes.** The shipped disc-centred set has 67 features; 36
  compute on these shapes and 8 have a value to be checked against. The rest need a fovea that means
  something, a photograph, or a quantity no shape defines.
- **Nothing about its defaults on a real frame.** Its optic disc class resizes to 1024 pixels
  whatever the retina's resolution, which the adapter works around — see the results page §4.
- **Nothing that selects.** No implementation passes or fails here.